# Selección y detalle de captura

El mismo cálculo independiente produce media 2.5 y varianza poblacional 1.25.
La sesión mínima conserva identidad y estado de dos operaciones científicas;
la detallada añade ocho inspecciones y referencias nativas. La configuración
se decide una vez, sin cambiar los cuerpos científicos.

Las exclusiones cuentan invocaciones, no resultados observados. Hasta un cierre
persistido esos recuentos son desconocidos. El modo mínimo omite las referencias
a entradas y salidas: sirve para seguir ejecución, pero ofrece menos procedencia
científica. Ninguno de los modos garantiza replay ni captura llamadas arbitrarias.
Los tiempos son observaciones locales y no se exige una razón de velocidades.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("experiments").resolve()))
import recorda
from consumer import REFERENCE_ADAPTERS
from run_selection import POLICIES, calculate, run

import recorda_lab

workspace = Path("selection-example")
workspace.mkdir()
samples = [1, 2, 3, 4]
expected = recorda_lab.summarize(samples)

In [ ]:
minimal_handle = recorda.start(
    "minimal",
    path=workspace / "minimal.jsonl",
    capture_policy=POLICIES["minimal"],
    reference_adapters=REFERENCE_ADAPTERS,
)

In [ ]:
minimal_native = calculate(samples, inspections=8)
assert minimal_native == expected
assert minimal_handle.record.status == "incomplete"
assert minimal_handle.record.coverage["excluded_boundaries"] is None

In [ ]:
minimal = minimal_handle.stop()
assert minimal.status == "succeeded" and len(minimal.operations) == 2
assert minimal.coverage["excluded_boundaries"]["by_profile"] == [
    {"profile": "inspection", "calls": 8}
]
minimal.coverage

In [ ]:
detailed_handle = recorda.start(
    "detailed",
    path=workspace / "detailed.jsonl",
    capture_policy=POLICIES["detailed"],
    reference_adapters=REFERENCE_ADAPTERS,
)

In [ ]:
detailed_native = calculate(samples, inspections=8)
assert detailed_native == minimal_native == expected

In [ ]:
detailed = detailed_handle.stop()
assert detailed.status == "succeeded" and len(detailed.operations) == 10
assert detailed.operations[0]["outputs"]["return"]["identifier"] == expected.identifier
comparison = {
    "minimal": {"operations": len(minimal.operations), "bytes": minimal_handle.path.stat().st_size},
    "detailed": {
        "operations": len(detailed.operations),
        "bytes": detailed_handle.path.stat().st_size,
    },
}
assert comparison["minimal"]["bytes"] < comparison["detailed"]["bytes"]
comparison

In [ ]:
measurements = run(workspace / "measurements", repetitions=3, inspections=8)
assert measurements["failures"] == {"minimal": "failed", "detailed": "failed"}
measurements["observations"]